# 06 Multilayer perceptron

Member D. `early_stopping` holds out a slice of the training set inside the fit. The project test set is still used only at the end.
Do not edit `src/common.py`, `data/`, or another member's notebook.


## Load the frozen split

Every model uses `common.train_test_data()`. That is the same 80/20 stratified split, seed 42.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import common

X_train, X_test, y_train, y_test = common.train_test_data()
print(common.split_summary(y_train, y_test))
print("features:", X_train.shape[1])


## Model

Change this cell when you tune. Leave the load cell and the scoring cell as they are.


In [ ]:
from sklearn.neural_network import MLPClassifier

model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    max_iter=300,
    early_stopping=True,
    random_state=common.SEED,
)


## Train and score

Cross-validation runs on the training set only. The test set is used once, after the fit.
This notebook writes `results/mlp_metrics.json`. Notebook 07 only reads that file.


In [ ]:
pipe = common.build_pipeline(model)
cv = common.cv_roc_auc(pipe, X_train, y_train)
pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)
y_prob = pipe.predict_proba(X_test)[:, 1]
holdout = common.holdout_metrics(y_test, y_pred, y_prob)
path = common.save_metrics("mlp", cv, holdout, y_train, y_test)

print("CV ROC-AUC:", cv["roc_auc_mean"], "+/-", cv["roc_auc_std"])
print("Holdout:", holdout)
print("Saved", path)


## Plots

Add the figures for your section here (ROC curve, confusion matrix, and the feature view that suits this model).
Save image files under `results/` if the report needs them. Clear notebook output before you commit.
